In [ ]:
import pandas as pd
from pathlib import Path
from machine_intelligence.utilities.data_handling import load_parquet_table

from machine_intelligence.utilities.data_quality import (
    get_parameter_coverage_ratio_df,
)


In [ ]:
cwd = Path.cwd()
cwd

In [ ]:
ds_path = cwd.parent / Path("datasets/sepsis_type_b_cohort/indexed/2024-07-17/df_all_aligned.parquet")

In [ ]:
df_ds = load_parquet_table(ds_path, backend="numpy_nullable")
df_ds


In [ ]:
df_ds_grouped = df_ds.groupby(by="patientunitstayid", group_keys=False)

df_variable_densities = get_parameter_coverage_ratio_df(df_grouped=df_ds_grouped, parameters=df_ds.columns)

restoring the proper index

In [ ]:
df_variable_densities.index = df_ds["patientunitstayid"].astype("int").unique().tolist()

In [ ]:
df_variable_densities

We will use heartrate as proxy variables to determine if the density of the sample is enough (threshold for discarding 25 %),
the heartrate should be the most abundant data point of a ICU stay

In [ ]:
dump_threshold = 0.25
df_patients_to_dump = df_variable_densities.loc[(df_variable_densities["heartrate"] < dump_threshold)]#or (df_variable_densities["heartrate"] > dump_threshold)]

In [ ]:
df_patients_to_dump.head()

In [ ]:
patient_ids_dump = df_patients_to_dump.index.tolist()
patient_ids_dump

In [ ]:

df_ds["patientunitstayid"] = df_ds["patientunitstayid"].astype("int")

df_ds[~df_ds["patientunitstayid"].isin(patient_ids_dump)]


-> Pretty big reduction in rows 